In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from ripser import ripser
from persim import PersistenceImager
from SALib.sample import saltelli
from SALib.analyze import sobol

# =========================================================
# Duffing oscillator
# =========================================================
def duffing(t, y, zeta, alpha, F, omega):
	x, v = y
	dxdt = v
	dvdt = -2*zeta*v - x - alpha*x**3 + F*np.cos(omega*t)
	return [dxdt, dvdt]

# =========================================================
# Simulation
# =========================================================
def simulate(alpha, zeta, F, omega=1.2):
	t_span = (0, 200)
	t_eval = np.linspace(*t_span, 4000)

	sol = solve_ivp(
		duffing,
		t_span,
		[0, 0],
		t_eval=t_eval,
		args=(zeta, alpha, F, omega)
	)

	x = sol.y[0]
	v = sol.y[1]

	# remove transient
	idx = int(0.7 * len(x))
	return x[idx:], v[idx:]

# =========================================================
# Topological features (persistent homology)
# =========================================================
def compute_ph_features(x, v):
	point_cloud = np.vstack((x, v)).T

	diagrams = ripser(point_cloud, maxdim=1)['dgms']

	H1 = diagrams[1]

	if len(H1) == 0:
		return np.zeros(10)

	# lifetimes
	lifetimes = H1[:, 1] - H1[:, 0]
	lifetimes = np.sort(lifetimes)[::-1]

	# pad to fixed size
	k = 10
	features = np.zeros(k)
	features[:min(k, len(lifetimes))] = lifetimes[:k]

	return features

# =========================================================
# Spectral features (optional but useful)
# =========================================================
def compute_fft_features(x):
	fft = np.abs(np.fft.rfft(x))
	fft = fft / np.max(fft)

	# count significant peaks
	threshold = 0.1
	return np.sum(fft > threshold)

# =========================================================
# Parameter sweep
# =========================================================
alpha_vals = np.linspace(0, 5, 30)
F_vals = np.linspace(0.1, 1.0, 30)

zeta = 0.05

features = []
params = []

print("Running parameter sweep...")

for i, alpha in enumerate(alpha_vals):
	for j, F in enumerate(F_vals):
		print(f"alpha={alpha:.2f}, F={F:.2f}")

		x, v = simulate(alpha, zeta, F)

		ph_feat = compute_ph_features(x, v)
		fft_feat = compute_fft_features(x)

		feat = np.concatenate([ph_feat, [fft_feat]])

		features.append(feat)
		params.append([alpha, F])

features = np.array(features)
params = np.array(params)

# =========================================================
# Clustering (regime discovery)
# =========================================================
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=0)
labels = kmeans.fit_predict(features)

# =========================================================
# Latent visualization
# =========================================================
pca = PCA(n_components=2)
latent = pca.fit_transform(features)

plt.figure()
plt.scatter(latent[:, 0], latent[:, 1], c=labels, cmap='tab10')
plt.title("Latent space (PCA)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

# =========================================================
# Phase diagram
# =========================================================
phase_map = labels.reshape(len(alpha_vals), len(F_vals))

plt.figure()
plt.imshow(
	phase_map,
	extent=[F_vals.min(), F_vals.max(), alpha_vals.min(), alpha_vals.max()],
	aspect='auto',
	origin='lower',
	cmap='tab10'
)
plt.colorbar(label="Regime")
plt.xlabel("F")
plt.ylabel("alpha")
plt.title("Learned Phase Diagram")
plt.show()

# =========================================================
# Sobol analysis (optional)
# =========================================================
print("Running Sobol analysis...")

def sobol_model(X):
	Y = []
	for row in X:
		alpha, zeta, F = row
		x, v = simulate(alpha, zeta, F)
		y = np.max(np.abs(x))
		Y.append(y)
	return np.array(Y)

problem = {
	'num_vars': 3,
	'names': ['alpha', 'zeta', 'F'],
	'bounds': [
		[0.0, 5.0],
		[0.01, 0.2],
		[0.1, 1.0]
	]
}

param_values = saltelli.sample(problem, 256)
Y = sobol_model(param_values)

Si = sobol.analyze(problem, Y)

print("\nSobol First-order:")
for name, val in zip(problem['names'], Si['S1']):
	print(f"{name}: {val:.3f}")

print("\nSobol Total-order:")
for name, val in zip(problem['names'], Si['ST']):
	print(f"{name}: {val:.3f}")

ModuleNotFoundError: No module named 'ripser'